## Q1. Embedding a query

Embed the following query:

> How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value
(`v[0]`)?

- -0.31
- -0.02
- 0.12
- 0.44

In [6]:
from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"

v= embed.encode(q1)
v[0]

-0.02058200593003704

## Loading the data

We pull the lesson pages from the course repository, the same way as in
homework 1. We pin to commit `8c1834d` so everyone works with the same
data.

Each document is a dictionary with a `filename` and `content`, and there
are 72 pages.

In [8]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

## Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product between two
of them is their cosine similarity.

Take the page `02-vector-search/lessons/07-sqlitesearch-vector.md`, embed
its `content`, and compute the cosine similarity with the query vector
from Q1. What do you get?

- 0.07
- 0.37
- 0.68
- 0.92

In [25]:
docs = [doc for doc in documents if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"]
docs_by_filename = {doc['filename']: doc["content"] for doc in docs}
dv = embed.encode(docs_by_filename["02-vector-search/lessons/07-sqlitesearch-vector.md"])
v.dot(dv)


0.3610702814461231

## Q3. Chunking and search by hand

A full page covers several topics, which waters down its embedding.

We chunk the pages the same way as in homework 1:

```python
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
```

We embed every chunk's `content` with `encode_batch`, stack the vectors
into a matrix `X`, and score the Q1 query against all chunks:

```python
scores = X.dot(v)
```

Which file does the highest-scoring chunk belong to (its `filename`)?

- `02-vector-search/lessons/03-embeddings-dataset.md`
- `02-vector-search/lessons/06-rag-vector.md`
- `02-vector-search/lessons/07-sqlitesearch-vector.md`
- `02-vector-search/lessons/09-onnx-embedder.md`

In [36]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

from tqdm.auto import tqdm
import numpy as np

X = []

for i in tqdm(range(0, len(chunks))):
    batch_vectors = embed.encode_batch([chunks[i]['content']])
    X.extend(batch_vectors)

X = np.array(X)

scores = X.dot(v)
best_idx = scores.argmax()
chunks[best_idx]['filename']

  0%|          | 0/295 [00:00<?, ?it/s]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

## Q4. Vector search with minsearch

We've done vector search by hand, which is good for learning, but it's not
what we do in practice. In practice we use libraries.

Let's use `VectorSearch` from minsearch and run a search for the following
query:

> What metric do we use to evaluate a search engine?

Which file is the `filename` of the first result?

- `02-vector-search/lessons/04-vector-search.md`
- `04-evaluation/lessons/05-search-metrics.md`
- `04-evaluation/lessons/13-llm-as-judge.md`
- `05-monitoring/lessons/04-metrics.md`

In [46]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

## Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index
the same chunks with `Index` from minsearch. Use `content` as a
text field.

Run both searches for this query:

> How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the
vector results but not in the text results?

- `02-vector-search/lessons/01-intro.md`
- `02-vector-search/lessons/02-embeddings.md`
- `02-vector-search/lessons/08-pgvector.md`
- `03-orchestration/lessons/05-rag.md`

In [47]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

query = "How do I store vectors in PostgreSQL?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

for i in range(0, len(results)):
    print(results[i]["filename"])

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [61]:
from minsearch import Index

query = "How do I store vectors in PostgreSQL?"

chunks = chunk_documents(documents, size=2000, step=1000)

index = Index(text_fields=["content"],keyword_fields=["filename"])
index.fit(chunks)

search_results = index.search(query,num_results=5)

for i in range(0, len(search_results)):
    print(search_results[i]["filename"])


02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


## Q6. Hybrid search

Both vector and text search have their strengths and weaknesses. Vector
search matches by meaning, so it finds relevant pages even when they use
words different from the query. But it can miss exact terms like names,
codes, or rare keywords. Text search is the opposite: it nails exact words
but misses paraphrases and synonyms.

We don't have to pick one or the other - we can use both and merge their
results. This approach is called "hybrid search".

Each search produces its own ranked list, so we need a way to combine them
into one. In this homework we use Reciprocal Rank Fusion (RRF). It ignores
the raw scores from each method, which live on different scales and aren't
directly comparable. Instead, it looks only at the position of each
document in each list.

Every document scores by its position (`rank`, starting at 0) in each
list, and we sum the scores across lists with a constant `k = 60`:

```text
RRF(d) = sum over lists of  1 / (k + rank(d))
```

"Sum over lists" means we go through every ranked list and, for each list
where the document appears, add its `1 / (k + rank)` contribution. A
document found by both searches collects a score from each list, while one
found by only a single search collects just one.

The constant `k` controls how much the exact rank matters. A larger `k`
flattens the gap between positions, so the difference between rank 0 and
rank 5 counts for less. A smaller `k` does the opposite: it sharpens that
gap, so being at the top of a list matters much more.

The value 60 comes from the original RRF paper and is the usual default.
You rarely need to tune it. Lower it when only the top results matter.
Raise it to reward documents that appear across many lists, even when they
never quite reach the top.

A document that ranks well in both lists ends up higher than one that's
only strong in a single list.

```python
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]
```

Now run the query `"How do I give the model access to tools?"`
with vector and text search and fuse the results with `rrf`:

```python
results = rrf([vector_results, text_results])
```

Which file is ranked first after RRF?

- `01-agentic-rag/lessons/01-intro.md`
- `01-agentic-rag/lessons/13-function-calling.md`
- `01-agentic-rag/lessons/14-agentic-loop.md`
- `01-agentic-rag/lessons/16-other-frameworks.md`

Notice that this file isn't first in either search on its own - it wins
because it ranks high in both.

In [62]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [65]:
query = "How do I give the model access to tools?"
query_vector = embed.encode(query)

vector_results = vindex.search(query_vector, num_results=5)
print("Vector results:")
for i in range(0, len(vector_results)):
    print(vector_results[i]["filename"])

text_results = index.search(query,num_results=5)
print("Text results:")
for i in range(0, len(text_results)):
    print(text_results[i]["filename"])

results = rrf([vector_results, text_results])
print("Hybrid results:")
for i in range(0, len(results)):
    print(results[i]["filename"])



Vector results:
01-agentic-rag/lessons/01-intro.md
04-evaluation/lessons/02-ground-truth.md
01-agentic-rag/lessons/16-other-frameworks.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md
Text results:
01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
04-evaluation/lessons/02-ground-truth.md
Hybrid results:
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/14-agentic-loop.md
04-evaluation/lessons/02-ground-truth.md
01-agentic-rag/lessons/16-other-frameworks.md


In [64]:
results


[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 